# 实验 26 — 端到端编排 + CI 风格 workflow

**目标**：把前面所有点串起来，模拟生产里一个完整的 ELT 流水线 + PR slim CI。

## 流水线长什么样

```
  ┌─ dlt ingest（incremental cursor + schema contract）
  │
  ├─ dbt source freshness（上游新鲜度门禁）
  │
  ├─ dbt snapshot（SCD2 维度历史）
  │
  ├─ dbt build（run + test）
  │
  └─ on-run-end: 写 audit + 发送告警
```

所有这些可以在一个 Python 脚本里跑（[orchestration.py](../orchestration.py)），也可以放到 Airflow / Dagster 的一个 DAG / Job 里。

In [ ]:
# 1) 完整跑一遍 orchestration.py
import subprocess
r = subprocess.run(['uv','run','python','orchestration.py'],
                   capture_output=True, text=True, cwd='..')
print(r.stdout[-2500:])
if r.returncode != 0: print('STDERR:', r.stderr[-800:])

## CI 风格 workflow（slim CI）

下面模拟 PR CI 的 4 步：

1. checkout PR 代码（在 git worktree 里，这里直接修改文件代替）
2. 下载 prod 的 manifest.json 作为 `--state`
3. 计算 `state:modified+` 找受影响模型
4. `dbt build --defer --state ... --select state:modified+`

In [ ]:
import os, shutil, subprocess
from pathlib import Path

def dbt(*a, target='dev'):
    r = subprocess.run(['uv','run','dbt','--target',target,*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.returncode, r.stdout + r.stderr

# Step 1: 假装 main 分支已经在 prod build 过，留有 manifest
rc, out = dbt('build','--exclude','snp_currencies', target='prod')
print('prod baseline build:', 'OK' if rc==0 else 'FAIL')

shutil.rmtree('../prod_state', ignore_errors=True)
os.makedirs('../prod_state')
shutil.copy('../dbt_project/target/manifest.json', '../prod_state/manifest.json')
print('prod manifest cached: ../prod_state/manifest.json')

In [ ]:
# Step 2: 模拟 PR：开发者改了 stg_ecb_rates.sql
stg = Path('../dbt_project/models/staging/stg_ecb_rates.sql')
orig = stg.read_text()
stg.write_text('-- PR change: rename column to be explicit\n' + orig)
print('PR diff applied.')

In [ ]:
# Step 3: slim CI build —— 只跑改的 + 下游，没改的从 prod defer
rc, out = dbt('build',
              '--select','state:modified+',
              '--defer','--state','../prod_state',
              '--exclude','snp_currencies')
print(out[-1800:])
print('RESULT:', 'OK' if rc==0 else 'FAIL')

In [ ]:
# Step 4: 比较 dev vs prod 结果（dbt_utils.equality 的简化版）
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
for t in ['stg_ecb_rates','fct_daily_rates','dim_currencies']:
    dev = con.execute(f'select count(*) from main.{t}').fetchone()[0]
    try:
        prod = con.execute(f'select count(*) from main_prod.{t}').fetchone()[0]
    except Exception:
        prod = None
    print(f'{t:25s} dev={dev:6}  prod={prod}')

In [ ]:
# 清理
stg.write_text(orig)
import shutil
shutil.rmtree('../prod_state', ignore_errors=True)

## 把上面所有点串成生产 SOP

**每日定时（cron / Airflow / Dagster schedule）：**
1. `dlt ingest`（带 incremental cursor）
2. `dbt source freshness`（如果 fail：alert，停止后续）
3. `dbt snapshot`
4. `dbt build`
5. 上传 `target/run_results.json` 到 S3，作为 CI defer 基线
6. on-run-end 写 audit 表（实验 21）

**每个 PR（GitHub Actions / GitLab CI）：**
1. 拉取上一个 prod 成功 run 的 `manifest.json`
2. 在 sandbox schema 跑：
   ```
   dbt build --defer --state ./prod_state --select state:modified+
   ```
3. 跑完销毁 sandbox schema

**事件响应：**
- freshness fail → 上游 ingestion 出问题
- test fail → 数据质量问题，看 `--store-failures` 的 audit 表
- on-run-end 失败列表 → 哪个模型挂了

## 思考题

- 把 orchestration.py 改成接受 `--target` 与 `--full-refresh` 参数。
- 用 GitHub Actions 写出这个 slim CI 的 yml（不用真的运行，列出关键步骤即可）。
- 你公司现在的 ELT 流水线哪一步最容易坏？最近一次半夜被叫醒是哪种问题？这套 SOP 里哪个环节能挡住它？